[backbone](https://github.com/open-mmlab/mmaction2/blob/main/mmaction/models/backbones/resnet3d_slowfast.py)

In [ ]:
from __future__ import annotations

import warnings
from collections import OrderDict
from typing import Optional

import torch
import torch.nn as nn


In [ ]:
class ResNet3dSlowFast(nn.Module):
    """Slowfast backbone
    This module is proposed in `SlowFast Networks for Video Recognition <https://arxiv.org/abs/1812.03982>`

    Args:
        pretrained (str): The file path to a pretrained model.
        resample_rate (int): A large temporal stride ``resample_rate`` on input frames. The actual resample rate is calculated by
            multiplying the ``interval`` in ``SampleFrames`` in the data-processing pipeline with ``resample_rate``, equivalent to `\tau` 
            in the paper, i.e., it processes only one out of ``\tau=resample_rate*interval`` frames. Defaults to 8.
        speed_ratio (int): Speed ratio indicating the ratio between time dimension of fast and slow pathway, corresponding to the `alpha`
            in the paper. Default to 8.
        channel_ratio (int): Reduce the channel number of fast pathway by ``channel_ratio``, corresponding to `\beta` in  the paper. 
            Default to 8.
        slow_pathway (dict): Configuration of slow branch. Default to ``dict(type='resnet3d', depth=50, pretrained=None, lateral=False,
            base_channels=8, conv1_kernel=(5,7,7), conv1_stride_t=1, pool1_stride_t=1)``
        fast_pathway (dict): Configuration of fast branch. Default to ``dict(type='resnet3d', depth=50, pretrained=None, lateral=False,
            base_channels=8, conv1_kernel=(5,7,7), conv1_stride_t=1, pool1_stride_t=1)``
        init_cfg (dict|list[dict], optional): Initialization config dict. Default to None
    """
    def __init__(self, pretrained:str|None=None, resample_rate:int=8, speed_ratio:int=8, channel_ratio:int=8, 
                 slow_pathway:dict=dict(type='resnet3d', depth=50, pretrained=None, lateral=True, conv1_kernel=(1,7,7),
                                       conv1_stride_t=1, pool1_stride_t=1, inflate=(0,0,1,1)),
                 fast_pathway:dict=dict(type='resnet3d', depth=50, pretrained=None, lateral=False, base_channels=8, 
                                        conv1_kernel=(5,7,7), conv1_stride_t=1, pool1_stride_t=1),
                 init_cfg:dict|None=None
                )->None:
        super().__init__()
        self.pretrained=pretrained
        self.resample_rate=resample_rate
        self.speed_ratio=speed_ratio
        self.channel_ratio=channel_ratio

        if slow_pathway['lateral']:
            slow_pathway['speed_ratio']=speed_ratio
            slow_pathway['channel_ratio']=channel_ratio

        # self.slow_path=build_pathway(slow_pathway)
        # self.fast_path=build_pathway(fast_pathway)

In [2]:
kwargs={'type': 'ResNet3dSlowFast',
 'pretrained': None,
 'resample_rate': 8,
 'speed_ratio': 8,
 'channel_ratio': 8,
 'slow_pathway': {'type': 'resnet3d',
  'depth': 50,
  'pretrained': None,
  'lateral': True,
  'conv1_kernel': (1, 7, 7),
  'dilations': (1, 1, 1, 1),
  'conv1_stride_t': 1,
  'pool1_stride_t': 1,
  'inflate': (0, 0, 1, 1),
  'norm_eval': False},
 'fast_pathway': {'type': 'resnet3d',
  'depth': 50,
  'pretrained': None,
  'lateral': False,
  'base_channels': 8,
  'conv1_kernel': (5, 7, 7),
  'conv1_stride_t': 1,
  'pool1_stride_t': 1,
  'norm_eval': False}}

# kwargs.pop('type')
slow_pathway=kwargs['slow_pathway']
fast_pathway=kwargs['fast_pathway']